# Note it was found that I could not easily dynamically pull the notebook name so if you update the notebook update the following code:

In [1]:
# Run the following once and once only when setting up jupyter notebooks for the 13-XX-XX series tests. Comment out afterwards.
# sudo apt install -y python3-dmidecode

# The following has to be done to ensure you are using the NVIDIA GPU.
# Be in base repo directory
# wget https://developer.download.nvidia.com/compute/cuda/12.4.1/local_installers/cuda_12.4.1_550.54.15_linux.run
# sudo sh cuda_12.4.1_550.54.15_linux.run
#
# conda env create -f environment.yml
#
# CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python
#
# export LD_LIBRARY_PATH=/usr/local/cuda-12.4/lib64:$LD_LIBRARY_PATH
#
# Test via:
# python -c "from llama_cpp import Llama; Llama(model_path='{FULL_MODEL_PATH_HERE}', n_gpu_layers=1, verbose=True)" 2>&1 | grep "Device"
#
# Example Output:
# Device 0: NVIDIA GeForce RTX 3090, compute capability 8.6, VMM: yes

In [2]:
nb_name = "MAT-13-02-level-2-notebook"

In [3]:
from datetime import datetime
import os
from pathlib import Path
import subprocess
import sys
import time

# Get project base directory (one level up from current working directory)
base_dir = Path.cwd().parent

base_application_dir = base_dir / "base_application"

# Convert to absolute string path
base_application_dir = str(base_application_dir.resolve())

# Add to Python path if not already present
if base_application_dir not in sys.path:
    sys.path.append(base_application_dir)

print("base_application added to PATH:")
print(base_application_dir)

base_application added to PATH:
/home/flaniganp/Documents/my-python-buddy/base_application


In [4]:
from llama_cpp import Llama
from views_chat_utilities import get_cleaned_code_response

In [5]:
# Load local fine-tuned model (GGUF) once at startup
model_name = "llama-2-7b-143k-codeAlpaca-q4_K_M-2025-10-30_1326.gguf"
full_model_path = os.path.join(base_dir, "models", model_name)
llm = Llama(
    model_path=os.path.join(base_dir, "models", model_name),
    n_ctx=4096,  # expand to match the model’s training context
    n_gpu_layers=-1,  # keep all layers on GPU (auto-fit)
    verbose=False,  # disables most llama.cpp logs
)

In [6]:
# Check that llama-cpp-python (in notebook) can find the gpu
subprocess.run(
    ["bash", "-c", f"python -c \"from llama_cpp import Llama; Llama(model_path='{full_model_path}', n_gpu_layers=1, verbose=True)\" 2>&1 | grep 'Device'"],
    check=True,
)

  Device 0: NVIDIA GeForce RTX 3090, compute capability 8.6, VMM: yes


CompletedProcess(args=['bash', '-c', 'python -c "from llama_cpp import Llama; Llama(model_path=\'/home/flaniganp/Documents/my-python-buddy/models/llama-2-7b-143k-codeAlpaca-q4_K_M-2025-10-30_1326.gguf\', n_gpu_layers=1, verbose=True)" 2>&1 | grep \'Device\''], returncode=0)

In [7]:
questions = [
    "Code: How do I filter even numbers from a list using a list comprehension?",
    "Code: How do I find the common elements between two lists in Python?",
    "Code: How do I remove all vowels from a string in Python?",
    "Code: How do I count how many times each word appears in a string?",
    "Code: How do I check if a string is a palindrome, ignoring case and spaces?",
    "Code: How do I get the current date and time in Python?",
    "Code: How do I find the factorial of a number using recursion in Python?",
    "Code: How do I merge two dictionaries into one in Python?",
    "Code: How do I find all unique words in a sentence, ignoring case and punctuation?",
    "Code: How do I find the most frequent element in a list without using the collections module?",
    "Code: How do I determine if two strings are anagrams of each other in Python?",
]

In [8]:
# Setup output directory and file
base_dir = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
results_dir = base_dir / "test_results"
results_dir.mkdir(exist_ok=True)

timestamp = datetime.now().strftime("%Y_%m_%d_%H_%M_%S")
output_file = results_dir / f"{nb_name}_{timestamp}.txt"

execution_times = []  # store all durations

# Begin loop over questions
with open(output_file, "w", encoding="utf-8") as f:
    for i, user_question in enumerate(questions, start=1):
        print(f"\nQuestion {i}: {user_question}")
        f.write(f"\nQuestion {i}: {user_question}\n")

        # Measure time
        start_time = time.time()
        cleaned_response, prompt_tokens, max_new_tokens = get_cleaned_code_response(llm, user_question)
        end_time = time.time()
        duration = end_time - start_time
        execution_times.append(duration)

        #  Display and record results
        print(f"Cleaned Response:\n{cleaned_response}")
        print("-" * 88)

        f.write(f"Cleaned Response:\n{cleaned_response}\n")
        f.write(f"Prompt Tokens: {prompt_tokens}\n")
        f.write(f"Max New Tokens: {max_new_tokens}\n")
        f.write(f"Time Taken: {duration:.2f} seconds\n")
        f.write("Is result correct? If not, one sentence as to why:\n\n")
        f.write("-" * 88 + "\n")

    # Compute and record summary stats
    if execution_times:
        low_time = min(execution_times)
        high_time = max(execution_times)
        avg_time = sum(execution_times) / len(execution_times)

        summary = (
            f"\nExecution Time Summary:\n"
            f"Lowest Time:  {low_time:.2f} seconds\n"
            f"Highest Time: {high_time:.2f} seconds\n"
            f"Average Time: {avg_time:.2f} seconds\n"
        )

        print(summary)
        f.write(summary)

print(f"\nAll results recorded to: {output_file}")


Question 1: Code: How do I filter even numbers from a list using a list comprehension?
Cleaned Response:
numbers = [1, 2, 3, 4, 5]
filtered_numbers = [num for num in numbers if num % 2 == 0]
print(filtered_numbers)

----------------------------------------------------------------------------------------

Question 2: Code: How do I find the common elements between two lists in Python?
Cleaned Response:
def find_common_elements(list1, list2):
    common_elements = []
    for element in list1:
        if element in list2:
            common_elements.append(element)
    return common_elements

----------------------------------------------------------------------------------------

Question 3: Code: How do I remove all vowels from a string in Python?
Cleaned Response:
def remove_vowels(s):
    vowels = "aeiouAEIOU"
    return "".join([char for char in s if char not in vowels])

----------------------------------------------------------------------------------------

Question 4: Code: How 